In [16]:
import os

In [17]:
dn2bipartite = dict()
_k = 10
model_name = 'tracktor'  # sort, deepsort, uma
dataset = ['training/0019', 'testing/0019', 'testing/0022', 'testing/0023', 
           'testing/0024', 'testing/0025', 'testing/0026', 'testing/0028']
for dn in dataset:
    root_dir = f'../../storage/results/kitti/{dn}/'.format(dn)
    root_dir += 'feats-raw-filtered_triplet_mrg005_wt10_wx05/'
    root_dir += f'reid-feat-person-images-filtered/{model_name}-osnet_x1_0-pairwise-avg-mot3/'
    tid_all = os.listdir(root_dir)
    tid_all = sorted([int(tid) for tid in tid_all])
    # print('tid_all:', tid_all)
    f_template = root_dir + '/{}/frames/'
    tid2fid = {}  # {0: [1, 11, 21, ...], ...}
    for tid in tid_all:
        f_all = os.listdir(f_template.format(tid))
        f_all = sorted([int(fn.split('.')[0]) for fn in f_all])
        tid2fid[tid] = [f_all[idx * (len(f_all) // _k)] for idx in range(_k + 1)
                        if idx * (len(f_all) // _k) < len(f_all)]
    bipartite_tid, bipartite_fid = 'var bipartite_tid = [', '            var bipartite_fid = ['
    for tid1 in tid_all:
        for tid2 in tid_all:
            if tid1 < tid2:
                bipartite_tid += '[{}, {}], '.format(tid1, tid2)
                bipartite_fid += '[{}, {}], '.format(tid2fid[tid1], tid2fid[tid2])
    bipartite_tid += '];'
    bipartite_fid += '];'
    dn2bipartite[dn] = [bipartite_tid, bipartite_fid]

In [18]:
html_templete = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>%s</title>
</head>
<body>
    <script>
        function myFunction(dn, model_name, images1, images2, info, bipartite_tid, bipartite_fid, index) {
            info.innerHTML += '<br>' + bipartite_tid[index][0] + ' vs. ' + bipartite_tid[index][1];
            for (let i = 0; i < bipartite_fid[index][0].length; i++) {
                images1[i].src = '../../../storage/results/kitti/' + dn
                                 + 'feats-raw-filtered_triplet_mrg005_wt10_wx05/'
                                 + 'reid-feat-person-images-filtered/'
                                 + model_name + '-osnet_x1_0-pairwise-avg-mot3/'
                                 + bipartite_tid[index][0] + '/frames/' + bipartite_fid[index][0][i] + '.png';
            }
            for (let i = 0; i < bipartite_fid[index][0].length; i++) {
                images2[i].src = '../../../storage/results/kitti/' + dn
                                 + 'feats-raw-filtered_triplet_mrg005_wt10_wx05/'
                                 + 'reid-feat-person-images-filtered/'
                                 + model_name + '-osnet_x1_0-pairwise-avg-mot3/'
                                 + bipartite_tid[index][1] + '/frames/' + bipartite_fid[index][1][i] + '.png';
            }
        };
        window.onload = function(){
            var info = document.getElementById("info");
            var images1 = document.getElementsByClassName("images1");
            var images2 = document.getElementsByClassName("images2");
            var back_bt = document.getElementById("back_bt");
            var match_bt = document.getElementById("match_bt");
            var no_match_bt = document.getElementById("no_match_bt");
            var dn = '%s/';
            var model_name = 'tracktor';
            info.innerHTML += '<p>' + dn + '<p>' + model_name;
            %s
            %s
            var index = 0;
            start_bt.onclick = function(){
                myFunction(dn, model_name, images1, images2, info, bipartite_tid, bipartite_fid, index);
            };
            back_bt.onclick = function(){
                index--;
                myFunction(dn, model_name, images1, images2, info, bipartite_tid, bipartite_fid, index);
            };
            match_bt.onclick = function(){
                info.innerHTML += ' match';
                index++;
                myFunction(dn, model_name, images1, images2, info, bipartite_tid, bipartite_fid, index);
            };
            no_match_bt.onclick = function(){
                index++;
                myFunction(dn, model_name, images1, images2, info, bipartite_tid, bipartite_fid, index);
            };
        };
    </script>
    <button id='start_bt' value="start"> start </button>
    <div id="image_set1">
        <img class="images1" src="" height="200">
        <img class="images1" src="" height="200">
        <img class="images1" src="" height="200">
        <img class="images1" src="" height="200">
        <img class="images1" src="" height="200">
        <img class="images1" src="" height="200">
        <img class="images1" src="" height="200">
        <img class="images1" src="" height="200">
        <img class="images1" src="" height="200">
        <img class="images1" src="" height="200">
        <img class="images1" src="" height="200">
    </div>
    <div id="image_set2">
        <img class="images2" src="" height="200">
        <img class="images2" src="" height="200">
        <img class="images2" src="" height="200">
        <img class="images2" src="" height="200">
        <img class="images2" src="" height="200">
        <img class="images2" src="" height="200">
        <img class="images2" src="" height="200">
        <img class="images2" src="" height="200">
        <img class="images2" src="" height="200">
        <img class="images2" src="" height="200">
        <img class="images2" src="" height="200">
    </div>
    <button id='back_bt' style="height:40px;width:200px"> back </button>
    <button id='match_bt' style="height:40px;width:200px"> match </button>
    <button id='no_match_bt' style="height:40px;width:200px"> not match </button>
    <p id="info"> info </p>
</body>
</html>
"""

In [19]:
labeling_page = 'labeling_page/'
if not os.path.exists(labeling_page):
    os.mkdir(labeling_page)
for dn in dataset:
    file_name = "-".join(dn.split('/'))
    with open(os.path.join(labeling_page, "%s.html" % file_name), "w") as out_file:
        final_html = html_templete % (file_name, dn, dn2bipartite[dn][0], dn2bipartite[dn][1])
        out_file.write(final_html)